In [13]:
# Upgrade core data and visualization libraries to their latest versions
%pip install --upgrade pandas numpy seaborn plotly 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Kiva Lending Patterns - Assessed Visualizations

Three charts selected as the assessed deliverables for the final submission.

In [14]:
import os
import pandas as pd
import numpy as np
import plotly.express as px

CLEANED_PATH = os.path.join('..', 'data', 'kiva_cleaned.csv')
dfc = pd.read_csv(CLEANED_PATH, index_col=0)
print(f"Cleaned dataset loaded: {dfc.shape[0]:,} rows, {dfc.shape[1]} columns")

Cleaned dataset loaded: 645,626 rows, 14 columns


### Assessed Chart 1 - Loan Count by Country

**Objective:** Identify which countries have the most Kiva loan records.

**What to look for:** Geographic concentration and the gap between the top country and the rest.

**Limitation:** Count reflects number of loans, not monetary volume.
Country population and economic size affect counts.

In [ ]:
TOP_N_COUNTRIES = 10
country_counts = dfc['country'].value_counts().head(TOP_N_COUNTRIES).reset_index()
country_counts.columns = ['country', 'count']
country_counts['pct'] = (country_counts['count'] / len(dfc) * 100).round(1)

fig_a1 = px.bar(
    country_counts,
    x='count',
    y='country',
    orientation='h',
    text=country_counts['pct'].map('{:.1f}%'.format),
    custom_data=['pct'],
    title=f'Top {TOP_N_COUNTRIES} Countries by Number of Kiva Loans',
    labels={'country': 'Country', 'count': 'Number of Loans'}
)
fig_a1.update_traces(
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Number of loans: %{x:,}<br>'
        '% of total number of loans: %{customdata[0]:.1f}%'
        '<extra></extra>'
    )
)
fig_a1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_a1.show()

### Assessed Chart 2 - Sector Distribution for Top N Countries (100% stacked bar)

**Objective:** Show which sectors account for the largest share of loans in the countries
with the highest lending activity.

**What to look for:** Countries where a single sector dominates vs. countries with a
diversified sector mix. Differences in sector priorities between top-lending countries.

**Configurable parameter:** `TOP_N_COUNTRIES` controls how many countries appear.
Increase to see more context; decrease to focus on the biggest lenders.

**Limitation:** Proportions show sector composition, not monetary volume.
Sample sizes are shown in country labels.

In [16]:
# Number of top countries to display -- increase or decrease as needed
TOP_N_COUNTRIES = 10

top_countries = dfc['country'].value_counts().head(TOP_N_COUNTRIES).index.tolist()
top_df = dfc[dfc['country'].isin(top_countries)].copy()

sector_by_country = (
    top_df.groupby(['country', 'sector'])
    .size()
    .reset_index(name='count')
)
country_totals = top_df.groupby('country').size().reset_index(name='total')
sector_by_country = sector_by_country.merge(country_totals, on='country')
sector_by_country['pct'] = sector_by_country['count'] / sector_by_country['total'] * 100
sector_by_country['country_label'] = sector_by_country.apply(
    lambda r: f"{r['country']} ({r['total']:,})", axis=1
)

country_order = (
    top_df.groupby('country').size()
    .sort_values(ascending=False)
    .index.tolist()
)
label_order = [
    sector_by_country.loc[sector_by_country['country'] == c, 'country_label'].iloc[0]
    for c in country_order
]

# Sectors sorted by descending overall share across the full dataset
sector_order = (
    dfc['sector'].value_counts(normalize=True)
    .sort_values(ascending=False)
    .index.tolist()
)

fig_a2 = px.bar(
    sector_by_country,
    x='pct',
    y='country_label',
    color='sector',
    orientation='h',
    custom_data=['pct', 'country'],
    title=f'Sector Distribution for Top {TOP_N_COUNTRIES} Countries by Loan Count',
    labels={
        'pct': '% of Loans',
        'country_label': 'Country',
        'sector': 'Sector'
    },
    category_orders={'country_label': label_order, 'sector': sector_order}
)
fig_a2.update_traces(
    hovertemplate=(
        '<b>%{fullData.name}</b><br>'
        '% of total loans for %{customdata[1]}: %{customdata[0]:.1f}%'
        '<extra></extra>'
    )
)
fig_a2.update_layout(xaxis_ticksuffix='%')
fig_a2.show()

### Assessed Chart 3 - Loan Activity vs Typical Loan Size by Country (bubble scatter)

**Objective:** Explore whether countries with high lending activity tend to have smaller
or larger individual loans, and how much total funding each country attracts.

**X-axis:** Number of loans (lending activity).
**Y-axis:** Median loan amount in USD (typical loan size).
**Bubble size:** Total funded amount in USD (monetary weight of the lending activity).

**Hypothesis to test:** High-activity countries are micro-lending destinations with
smaller loan sizes, while low-activity countries may have larger individual loans.

**Configurable parameter:** `MIN_LOANS` is the minimum number of loans a country must
have to appear. Increase to focus on the most active countries; decrease to include
smaller markets.

**Limitation:** Country-level aggregation hides within-country variation. Bubble size
is total funded amount, not requested amount; partially funded loans affect this figure.

In [17]:
# Minimum number of loans for a country to appear in the chart -- adjust as needed
MIN_LOANS = 15000

country_bubble = (
    dfc.groupby('country')
    .agg(
        loan_count=('loan_amount', 'count'),
        median_loan_amount=('loan_amount', 'median'),
        total_funded_amount=('funded_amount', 'sum')
    )
    .reset_index()
)

# Dominant sector per country: the sector with the most loans in that country
dominant_sector = (
    dfc.groupby(['country', 'sector'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .drop_duplicates('country')
    .set_index('country')['sector']
)
country_bubble['dominant_sector'] = country_bubble['country'].map(dominant_sector)

country_bubble = (
    country_bubble[country_bubble['loan_count'] >= MIN_LOANS]
    .sort_values('loan_count', ascending=False)
    .reset_index(drop=True)
)

print(f"Countries shown (loan count >= {MIN_LOANS:,}): {len(country_bubble)}")
print(f"Countries excluded (below threshold): "
      f"{dfc['country'].nunique() - len(country_bubble)}")
print("Dominant sectors present:", sorted(country_bubble['dominant_sector'].unique()))

fig_a3 = px.scatter(
    country_bubble,
    x='loan_count',
    y='median_loan_amount',
    size='total_funded_amount',
    color='dominant_sector',
    text='country',
    custom_data=['total_funded_amount', 'dominant_sector'],
    title='Loan Activity vs Typical Loan Size by Country',
    labels={
        'loan_count': 'Number of Loans',
        'median_loan_amount': 'Median Loan Amount (USD)',
        'total_funded_amount': 'Total Funded (USD)',
        'dominant_sector': 'Dominant Sector'
    },
    size_max=60
)
fig_a3.update_traces(
    textposition='middle center',
    textfont_size=10,
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Dominant sector: %{customdata[1]}<br>'
        'Loan count: %{x:,}<br>'
        'Median loan amount: $%{y:,.0f}<br>'
        'Total funded: $%{customdata[0]:,.0f}'
        '<extra></extra>'
    )
)
fig_a3.update_layout(height=750)
fig_a3.show()

Countries shown (loan count >= 15,000): 9
Countries excluded (below threshold): 78
Dominant sectors present: ['Agriculture', 'Food', 'Retail', 'Services']
